# Colab Automation: Train + Drive Backup + Resume Check

This notebook automates the full training flow for collab_scripts:
1. Mount Google Drive
2. Clone/Bootstrap from GitHub
3. Optionally auto-pull dataset from Kaggle
4. Update pipeline_config.json paths
5. Run split -> train -> evaluate -> export
6. Verify backups are written to Drive
7. Check whether resume is ready and working

In [ ]:
from collections import deque
from pathlib import Path
import json
import os
import re
import subprocess

# ---- Required values ----
GITHUB_REPO_URL = "https://github.com/<org>/<repo>.git"
GITHUB_BRANCH = "main"
REPO_DIR = Path("/content/intruder_detection_system")

# Raw data location (class folders: fight/theft/intrusion/normal)
RAW_DATASET_DIR = Path("/content/raw_dataset")

# Optional Kaggle auto-pull for training data files
AUTO_PULL_KAGGLE = True
KAGGLE_DATASET_SLUG = "webadvisor/real-time-anomaly-detection-in-cctv-surveillance"
KAGGLE_CLEAN_TARGET = True

# Optional KaggleHub pandas preview (set file path to enable)
USE_KAGGLEHUB_PANDAS_PREVIEW = False
KAGGLEHUB_FILE_PATH = ""
KAGGLEHUB_SQL_QUERY = ""

# Paths for outputs/checkpoints
DATASET_DIR = Path("/content/dataset")
ARTIFACT_DIR = Path("/content/artifacts")
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/action_model_checkpoints")
LOCAL_CHECKPOINT_DIR = Path("/content/checkpoints")

# Execution toggles
RUN_PIPELINE_NOW = True
RUN_RESUME_COMMAND_CHECK = True

# Load Kaggle credentials from runtime env/Colab secrets only (do not hardcode in notebook).
kaggle_username = os.environ.get("KAGGLE_USERNAME", "")
kaggle_key = os.environ.get("KAGGLE_KEY", "")
if (not kaggle_username or not kaggle_key):
    try:
        from google.colab import userdata
        kaggle_username = kaggle_username or userdata.get("KAGGLE_USERNAME")
        kaggle_key = kaggle_key or userdata.get("KAGGLE_KEY")
    except Exception:
        pass
if kaggle_username and kaggle_key:
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key

def run_cmd(
    command,
    cwd=None,
    check=True,
    stream=False,
    capture_stream_output=False,
    stream_tail_lines=400,
):
    print("$", " ".join(command))
    if stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        tail = deque(maxlen=stream_tail_lines) if capture_stream_output else None
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            if tail is not None:
                tail.append(line)
        return_code = process.wait()
        stdout_text = "".join(tail) if tail is not None else ""
        if check and return_code != 0:
            raise subprocess.CalledProcessError(return_code, command, output=stdout_text)
        return subprocess.CompletedProcess(command, return_code, stdout=stdout_text, stderr="")

    result = subprocess.run(
        command,
        cwd=cwd,
        check=check,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-1500:])
    return result

print("Config initialized. Update GITHUB_REPO_URL before running next cells.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted.")

In [ ]:
if not (REPO_DIR / ".git").exists():
    run_cmd([
        "git",
        "clone",
        "--branch",
        GITHUB_BRANCH,
        "--single-branch",
        GITHUB_REPO_URL,
        str(REPO_DIR),
    ])
else:
    print(f"Repository already exists at {REPO_DIR}; pulling latest...")
    run_cmd(["git", "pull"], cwd=REPO_DIR)

bootstrap_candidates = [
    REPO_DIR / "backend/collab_scripts/bootstrap_colab.sh",
    REPO_DIR / "collab_scripts/bootstrap_colab.sh",
]
bootstrap_path = next((p for p in bootstrap_candidates if p.exists()), None)
if bootstrap_path is None:
    raise FileNotFoundError("Could not find bootstrap_colab.sh in cloned repo")

run_cmd(["bash", str(bootstrap_path), str(REPO_DIR)])
print("Clone + bootstrap complete.")

In [ ]:
config_candidates = [
    REPO_DIR / "backend/collab_scripts/pipeline_config.json",
    REPO_DIR / "collab_scripts/pipeline_config.json",
]
config_path = next((p for p in config_candidates if p.exists()), None)
if config_path is None:
    raise FileNotFoundError("pipeline_config.json not found in repo")

config = json.loads(config_path.read_text(encoding="utf-8"))
config.setdefault("paths", {})
config["paths"]["raw_dataset_dir"] = str(RAW_DATASET_DIR)
config["paths"]["dataset_dir"] = str(DATASET_DIR)
config["paths"]["checkpoint_dir"] = str(LOCAL_CHECKPOINT_DIR)
config["paths"]["drive_checkpoint_dir"] = str(DRIVE_CHECKPOINT_DIR)
config["paths"]["artifact_dir"] = str(ARTIFACT_DIR)
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print(f"Updated config: {config_path}")
print(json.dumps(config["paths"], indent=2))

In [ ]:
if USE_KAGGLEHUB_PANDAS_PREVIEW:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    if not KAGGLEHUB_FILE_PATH.strip():
        raise ValueError("Set KAGGLEHUB_FILE_PATH when USE_KAGGLEHUB_PANDAS_PREVIEW is True")

    load_kwargs = {}
    if KAGGLEHUB_SQL_QUERY.strip():
        load_kwargs["sql_query"] = KAGGLEHUB_SQL_QUERY

    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        KAGGLE_DATASET_SLUG,
        KAGGLEHUB_FILE_PATH,
        **load_kwargs,
    )
    print("KaggleHub DataFrame preview:")
    display(df.head())
else:
    print("KaggleHub pandas preview skipped.")

In [ ]:
expected_classes = ["fight", "theft", "intrusion", "normal"]
missing = [c for c in expected_classes if not (RAW_DATASET_DIR / c).exists()]

if AUTO_PULL_KAGGLE:
    if "<owner>/<dataset-slug>" in KAGGLE_DATASET_SLUG or not KAGGLE_DATASET_SLUG.strip():
        raise ValueError("Set KAGGLE_DATASET_SLUG before enabling AUTO_PULL_KAGGLE")
    print("AUTO_PULL_KAGGLE is enabled. Missing classes will be fetched by run script.")
else:
    if missing:
        raise FileNotFoundError(f"Missing class folders under {RAW_DATASET_DIR}: {missing}")
    print("Dataset class folders found.")

In [ ]:
if RUN_PIPELINE_NOW:
    run_script_candidates = [
        REPO_DIR / "backend/collab_scripts/colab_run_training.sh",
        REPO_DIR / "collab_scripts/colab_run_training.sh",
    ]
    run_script = next((p for p in run_script_candidates if p.exists()), None)
    if run_script is None:
        raise FileNotFoundError("colab_run_training.sh not found in repo")

    command = [
        "bash",
        str(run_script),
        str(REPO_DIR),
        "--config",
        str(config_path.relative_to(REPO_DIR)),
    ]

    if AUTO_PULL_KAGGLE:
        command.extend([
            "--kaggle-dataset",
            KAGGLE_DATASET_SLUG,
            "--raw-dataset-dir",
            str(RAW_DATASET_DIR),
        ])
        if KAGGLE_CLEAN_TARGET:
            command.append("--kaggle-clean")

    completed = run_cmd(command, cwd=REPO_DIR, stream=True)
else:
    print("RUN_PIPELINE_NOW is False. Skipping pipeline run.")

In [ ]:
drive_last = DRIVE_CHECKPOINT_DIR / "last.pt"
drive_best = DRIVE_CHECKPOINT_DIR / "best.pt"

print(f"Drive last.pt exists: {drive_last.exists()}")
print(f"Drive best.pt exists: {drive_best.exists()}")
if drive_last.exists():
    print(f"Drive last.pt size: {drive_last.stat().st_size} bytes")
    print(f"Drive last.pt mtime: {drive_last.stat().st_mtime}")
if drive_best.exists():
    print(f"Drive best.pt size: {drive_best.stat().st_size} bytes")
    print(f"Drive best.pt mtime: {drive_best.stat().st_mtime}")

In [ ]:
print(f"Local checkpoint exists: {(LOCAL_CHECKPOINT_DIR / 'last.pt').exists()}")
print(f"Drive checkpoint exists: {(DRIVE_CHECKPOINT_DIR / 'last.pt').exists()}")

if RUN_RESUME_COMMAND_CHECK:
    train_cmd = [
        "python",
        "-m",
        "collab_scripts.train_action_model",
        "--config",
        str(config_path.relative_to(REPO_DIR)),
        "--auto-resume",
    ]
    work_root = REPO_DIR / "backend" if (REPO_DIR / "backend/collab_scripts").exists() else REPO_DIR
    print("$", " ".join(train_cmd))
    process = subprocess.Popen(
        train_cmd,
        cwd=work_root,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    resume_epoch = None
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        if resume_epoch is None:
            match = re.search(r"Starting at epoch:\s*(\d+)", line)
            if match:
                resume_epoch = int(match.group(1))

    exit_code = process.wait()
    if resume_epoch is not None:
        print(f"Resume command output epoch: {resume_epoch}")
        print("Resume command check: WORKING" if resume_epoch > 0 else "Resume command check: FRESH START")
    else:
        print("Resume command check: Could not find resume marker in streamed output.")
    if exit_code != 0:
        print(f"Resume command exited with code {exit_code}")